[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/maryrobynbernabe/coral-sound-project-1-data-processing)

In [1]:
MAIN_PATH = '/kaggle/input/datasets/maryrobynbernabe/coral-sound-dataset/audio_dir'

In [2]:
import os
import pandas as pd

audio_files = []
# collect all audio file names in main path and convert it to a dataframe
for _,_,files in os.walk(MAIN_PATH):
  for rec in files:
    if rec.endswith('.wav'):
      audio_files.append(rec)

df_name = pd.DataFrame(audio_files, columns=['audio_file_name'])
df_name.head()

,audio_file_name
0,BoN10.2359R.1677983769.180905.1.12.wav
1,BoN11.0529R.1677983769.180906.4.45.wav
2,BoN11.1103D.1678278701.180906.4.32.wav
3,BaN12.0529R.1677983769.180907.5.46.wav
4,BaF5.1732R.1678278701.180831.2.12.wav


In [3]:
import librosa #
import pandas as pd
import numpy as np

#function to extract features from audion files
def extract_features(audio_path):

    # load the audio file
    y,sr = librosa.load(audio_path,mono=True) #load the audio file
    # extract features
    rmse = librosa.feature.rms(y=y)[0] #compute root-mean-square (RMS) value for each frame
    spec_cent = librosa.feature.spectral_centroid(y=y,sr=sr) #calculate spectral_centroid
    spec_bw = librosa.feature.spectral_bandwidth(y=y,sr=sr) #calculate spectral_bandwith
    rolloff = librosa.feature.spectral_rolloff(y=y,sr=sr) #calculate spectral_rolloff
    zcr = librosa.feature.zero_crossing_rate(y) #calculate zero crossing rate
    mfcc = librosa.feature.mfcc(y=y,sr=sr) #Mel frequency ceptral coefficients (mfcc)


    audio_dict={
        'RMSE':rmse.mean(),
        'SPECTRAL_CENTROID':spec_cent.mean(),
        'SPECTRAL_BANDWIDTH':spec_bw.mean(),
        'ROLLOFF':rolloff.mean(),
        'ZERO_CROSSING_RATE':zcr.mean()
    }

    #add the mfcc values
    for index,mfcc_set in enumerate(mfcc):
        feature_name=f"MFCC_FEATURE_{index}"
        audio_dict[feature_name]=np.mean(mfcc_set)

    return audio_dict

In [4]:
from tqdm.notebook import tqdm

# Extract audio features of each record
audio_features = []

for file_record in tqdm(df_name['audio_file_name'], desc="Processing Files"):
  file_path = os.path.join(MAIN_PATH, file_record) #get the file path
  features = extract_features(file_path) #extract features
  audio_features.append(features) #add extracted features to features list

print(len(features))

features_df = pd.DataFrame(audio_features) #convert audiio features to a dataframe
df_new = pd.concat([df_name.reset_index(drop=True), features_df], axis=1) #include file names in dataframe
df_new.head()

Processing Files:   0%|          | 0/262 [00:00<?, ?it/s]

25


,audio_file_name,RMSE,SPECTRAL_CENTROID,SPECTRAL_BANDWIDTH,ROLLOFF,ZERO_CROSSING_RATE,MFCC_FEATURE_0,MFCC_FEATURE_1,MFCC_FEATURE_2,MFCC_FEATURE_3,...,MFCC_FEATURE_10,MFCC_FEATURE_11,MFCC_FEATURE_12,MFCC_FEATURE_13,MFCC_FEATURE_14,MFCC_FEATURE_15,MFCC_FEATURE_16,MFCC_FEATURE_17,MFCC_FEATURE_18,MFCC_FEATURE_19
0,BoN10.2359R.1677983769.180905.1.12.wav,0.011000,3974.266047,3494.028931,8364.436921,0.003938,-508.000885,-70.216248,-5.753670,16.357086,...,9.112220,13.664226,8.291626,11.360783,3.441027,8.500538,4.896815,8.645019,4.216819,10.106939
1,BoN11.0529R.1677983769.180906.4.45.wav,0.010963,3616.489840,3552.171710,8205.517049,0.002235,-527.853882,-55.517910,11.541430,25.993612,...,3.957450,8.502945,5.018967,8.891335,3.383255,8.611948,5.192332,8.339452,3.200580,8.528194
2,BoN11.1103D.1678278701.180906.4.32.wav,0.013505,2203.188939,3327.939877,6814.896291,0.000203,-596.173828,-30.206333,32.680866,24.321074,...,4.004890,7.948227,3.422245,8.702552,2.499417,7.926836,3.409961,7.515243,3.444732,6.654953
3,BaN12.0529R.1677983769.180907.5.46.wav,0.010878,2849.557129,3414.181023,7438.905018,0.000900,-547.844788,-25.308653,21.758961,28.064041,...,1.155456,6.696609,3.105051,8.415883,3.191075,9.433007,4.989413,8.680643,3.263001,7.559288
4,BaF5.1732R.1678278701.180831.2.12.wav,0.013718,2819.472916,3277.748076,7121.031940,0.001491,-505.595825,-27.010841,5.536732,34.013489,...,4.580592,8.420740,4.733733,11.414305,4.753036,11.750537,7.183123,10.483688,6.316996,10.076572


In [5]:
# Add label class
labels = [
    'Healthy' if row.split('.')[1][4] == 'H' else
    'Degraded' if row.split('.')[1][4] == 'D' else
    'Restored'
    for row in df_new['audio_file_name']]

# Create the DataFrame for labels
df_label = pd.DataFrame({'labels': labels})

# Concatenate the DataFrames
df_main = pd.concat([df_new, df_label], axis=1)

df_main.head()

,audio_file_name,RMSE,SPECTRAL_CENTROID,SPECTRAL_BANDWIDTH,ROLLOFF,ZERO_CROSSING_RATE,MFCC_FEATURE_0,MFCC_FEATURE_1,MFCC_FEATURE_2,MFCC_FEATURE_3,...,MFCC_FEATURE_11,MFCC_FEATURE_12,MFCC_FEATURE_13,MFCC_FEATURE_14,MFCC_FEATURE_15,MFCC_FEATURE_16,MFCC_FEATURE_17,MFCC_FEATURE_18,MFCC_FEATURE_19,labels
0,BoN10.2359R.1677983769.180905.1.12.wav,0.011000,3974.266047,3494.028931,8364.436921,0.003938,-508.000885,-70.216248,-5.753670,16.357086,...,13.664226,8.291626,11.360783,3.441027,8.500538,4.896815,8.645019,4.216819,10.106939,Restored
1,BoN11.0529R.1677983769.180906.4.45.wav,0.010963,3616.489840,3552.171710,8205.517049,0.002235,-527.853882,-55.517910,11.541430,25.993612,...,8.502945,5.018967,8.891335,3.383255,8.611948,5.192332,8.339452,3.200580,8.528194,Restored
2,BoN11.1103D.1678278701.180906.4.32.wav,0.013505,2203.188939,3327.939877,6814.896291,0.000203,-596.173828,-30.206333,32.680866,24.321074,...,7.948227,3.422245,8.702552,2.499417,7.926836,3.409961,7.515243,3.444732,6.654953,Degraded
3,BaN12.0529R.1677983769.180907.5.46.wav,0.010878,2849.557129,3414.181023,7438.905018,0.000900,-547.844788,-25.308653,21.758961,28.064041,...,6.696609,3.105051,8.415883,3.191075,9.433007,4.989413,8.680643,3.263001,7.559288,Restored
4,BaF5.1732R.1678278701.180831.2.12.wav,0.013718,2819.472916,3277.748076,7121.031940,0.001491,-505.595825,-27.010841,5.536732,34.013489,...,8.420740,4.733733,11.414305,4.753036,11.750537,7.183123,10.483688,6.316996,10.076572,Restored


In [6]:
# Check for duplicate values
df_main.duplicated().sum()

np.int64(0)

In [7]:
# Check for null values
df_main.isnull().sum()

audio_file_name       0
RMSE                  0
SPECTRAL_CENTROID     0
SPECTRAL_BANDWIDTH    0
ROLLOFF               0
ZERO_CROSSING_RATE    0
MFCC_FEATURE_0        0
MFCC_FEATURE_1        0
MFCC_FEATURE_2        0
MFCC_FEATURE_3        0
MFCC_FEATURE_4        0
MFCC_FEATURE_5        0
MFCC_FEATURE_6        0
MFCC_FEATURE_7        0
MFCC_FEATURE_8        0
MFCC_FEATURE_9        0
MFCC_FEATURE_10       0
MFCC_FEATURE_11       0
MFCC_FEATURE_12       0
MFCC_FEATURE_13       0
MFCC_FEATURE_14       0
MFCC_FEATURE_15       0
MFCC_FEATURE_16       0
MFCC_FEATURE_17       0
MFCC_FEATURE_18       0
MFCC_FEATURE_19       0
labels                0
dtype: int64

In [8]:
# Save the DataFrame to a CSV file
df_main.to_csv('/kaggle/working/featurizedaudio.csv', index=False)